# Homework 3

## Note
\
This homework uses the pinned 2026 lead-scoring release in the course repository. The plan and report are available in cohorts/2026/data/.
Dataset \
\
In this homework, we will use the 2026 lead-scoring dataset. Download it from here.\
\
Or you can do it with wget:\
\
wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv\
\
In this dataset our desired target for classification task will be converted variable - has the client signed up to the platform or not.
Data preparation\
\
Check if the missing values are presented in the features.\
If there are missing values:
* For categorical features, replace them with 'NA'\
*  For numerical features, replace with with 0.0\


In [1]:
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

In [2]:
# 3.2 DOWNLOAD AND PREVIEW DATA
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv'

df = pd.read_csv(data)
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [3]:
# check for missing data
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [4]:
# CREATE LISTS OF COLUMNS BY CLASS
categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

binary = [
    'converted'
]

In [5]:
# FILL MISSING CATEGORICAL VALUES WITH 'NA'
df[categorical] = df[categorical].fillna('NA')

In [6]:
# FILL MISSING NUMERIC VALUES WITH '0'
# df.totalcharges = 
df[numerical] = df[numerical].fillna(0)

In [7]:
# VERIFY
df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

# Question 1
\
What is the most frequent observation (mode) for the column industry?\
*    NA
*    technology
*    healthcare
*    retail


In [8]:
# df['industry'].median()
df['industry'].mode().iloc[0]

# Q1 ANSWER IS 'technology'

'technology'

# Question 2

Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features. \
 \
What are the two features that have the biggest correlation? 
* interaction_count and lead_score
* number_of_courses_viewed and lead_score
* number_of_courses_viewed and interaction_count
* annual_income and interaction_count
 \
Only consider the pairs above when answering this question.

In [9]:
# IMPORT PACKAGES
from sklearn.model_selection import train_test_split

# SPLIT THE DATASET AND CREATE 'df_test'
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

# SPLIT 'df_full_train' AND CREATE 'df_val'
df_train, df_val = train_test_split(
    df_full_train,   # DATAFRAME TO SPLIT
    test_size=0.25,  # SPLIT PERCENTAGE
    random_state=42  # SEED
)


In [10]:
# RESET THE INDEX IN EACH DATASET
df_full_train = df_full_train.reset_index(drop=True)
df_train      = df_train.reset_index(drop=True)
df_val        = df_val.reset_index(drop=True)
df_test       = df_test.reset_index(drop=True)


In [11]:
# CREATE TARGET / LABEL VECTORS
# df_full_train = df_full_train.converted.values
y_train       = df_train.converted.values
y_val         = df_val.converted.values
y_test        = df_test.converted.values


In [12]:
# DROP TARGET / LABELS FROM FEATURE MATRIX
# del df_full_train['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted']


In [13]:
# DEBUG - CONFIRM 'converted' IS NOT PART OF DATASETS
# print(f"==== df_full_train ================ \r\n{df_train.isnull().sum()}\r\n")
print(f"==== df_train      ================ \r\n{df_train.isnull().sum()}\r\n")
print(f"==== df_val        ================ \r\n{df_val.isnull().sum()}\r\n")
print(f"==== df_test       ================ \r\n{df_test.isnull().sum()}\r\n")


==== df_train      ================ 
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
dtype: int64

==== df_val        ================ 
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
dtype: int64

==== df_test       ================ 
lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
dtype: int64



In [14]:
# DEBUG - VALIDATE SPLIT TOTALS
len(df), len(df_full_train), len(df_train), len(df_val), len(df_test)

(5000, 4000, 3000, 1000, 1000)

In [15]:
# CREATE CORRELATION MATRX FOR NUMERICAL FEATURES
corr_matrix = df_full_train[numerical].corrwith(df_full_train.converted)

#corr_matrix = df_full_train[numerical].corr()
corr_matrix

#  ANSWER: 
#    annual_income               0.188437
#    number_of_courses_viewed    0.393200
#    interaction_count           0.450273
#    lead_score                  0.482078
#    dtype: float64

annual_income               0.188437
number_of_courses_viewed    0.393200
interaction_count           0.450273
lead_score                  0.482078
dtype: float64

# Question 3

Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only. \
 \
Round the scores to 2 decimals using round(score, 2). \
 \
Which of these variables has the biggest mutual information score?
* industry
* location
* lead_source
* employment_status


In [16]:
# REMINDER: EARLIER WE CREATED THE FOLLOWIG LIST 
# categorical = [
#    'lead_source',
#    'industry',
#    'employment_status',
#    'location'
#]

# IMPORT PACKAGE
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score

In [17]:
# FUNCTION TO CALCULATE MUTUAL INFORMATION SCORE FOR ALL CATEGORICAL FEATURES
def mutual_info_converted_score(series):
    return mutual_info_score(series, df_full_train.converted)

In [18]:
# APPLY FUNCTION TO ALL CATEGORICAL FEATURES
mi = df_full_train[categorical].apply(mutual_info_converted_score)
mi.sort_values(ascending=False)

# ANSWER: lead_source has the biggest mutual information score
#    lead_source          0.034761
#    employment_status    0.020313
#    industry             0.002898
#    location             0.001050
#    dtype: float64


lead_source          0.034761
employment_status    0.020313
industry             0.002898
location             0.001050
dtype: float64

# Question 4
Now let's train a logistic regression. \
     \
Remember that we have several **categorical variables** in the dataset. Include them using one-hot encoding. 
1. Fit the model on the training dataset.
*  To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
* model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
 

2. Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
 \
What accuracy did you get? \
* 0.55
* 0.65
* 0.75
* 0.85


In [19]:
# CREATE LOGISTIC REGRESSION FUNCTIONS
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def logistic_regression(xi):
    score  = w0
    
    for j in range(len(w)):
        score = score + xi[j] * w[j]
        
    result = sigmoid(score)
    return result

In [20]:
# IMPORT PACKAGES
from sklearn.linear_model import LogisticRegression

In [21]:
# ONE-HOT ENCODING
#  IMPORT PACKAGE
from sklearn.feature_extraction import DictVectorizer

#  CREATE DICTIONARY VECTOR
dv         = DictVectorizer(sparse=False)

#  CREATE A TRAINING DICTIONARY
train_dict = df_train[categorical + numerical].to_dict(orient='records')

#  FIT LEARNS THE CATEGORIES FOR EACH FEATURE
#  TRANSFORM TURNS THE DICTIONARIES INTO A MATRIX
X_train    = dv.fit_transform(train_dict)

In [22]:
# TRAIN THE FEATURE MODEL
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# solver='lbfgs' is the default solver in newer version of sklearn
# for older versions, you need to specify it explicitly
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` problems (`n_classes >= 3`), all solvers except 'liblinear' minimize the full multinomial loss, 'liblinear' will raise an error.- 'newton-cholesky' is a good choice for `n_samples` >> `n_features * n_classes`, especially with one-hot encoded categorical features with rare categories. Be aware that the memory usage of this solver has a quadratic dependency on `n_features * n_classes` because it explicitly computes the full Hessian matrix.- For small datasets, 'liblinear' is a good choice, whereas 'sag' and 'saga' are faster for large ones;- 'liblinear' can only handle binary classification by default. To apply a one-versus-rest scheme for the multiclass setting one can wrap it with the :class:`~sklearn.multiclass.OneVsRestClassifier`... warning:: The choice of the algorithm depends on the penalty chosen (`l1_ratio=0` for L2-penalty, `l1_ratio=1` for L1-penalty and `0 < l1_ratio < 1` for Elastic-Net) and on (multinomial) multiclass support: ================= ======================== ====================== solver l1_ratio multinomial multiclass ================= ======================== ====================== 'lbfgs' l1_ratio=0 yes 'liblinear' l1_ratio=1 or l1_ratio=0 no 'newton-cg' l1_ratio=0 yes 'newton-cholesky' l1_ratio=0 yes 'sag' l1_ratio=0 yes 'saga' 0<=l1_ratio<=1 yes ================= ======================== ======================.. note:: 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`... seealso:: Refer to the :ref:`User Guide <Logistic_regression>` for more information regarding :class:`LogisticRegression` and more specifically the :ref:`Table <logistic_regression_solvers>` summarizing solver/penalty supports... versionadded:: 0.17 Stochastic Average Gradient (SAG) descent solver. Multinomial support in version 0.18... versionadded:: 0.19 SAGA solver... versionchanged:: 0.22 The default solver changed from 'liblinear' to 'lbfgs' in 0.22... versionadded:: 1.2 newton-cholesky solver. Multinomial support in version 1.6.",'liblinear'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l

In [23]:
# ENCODE THE VALIDATION DATASET THE SAME WAY AS THE TRAINING DATASET
#  NOTE: DO NOT 'fit' THE VALIDATION DATASET - WE WANT TO USE THE SAME CATEGORIES 
#        THAT WERE LEARNED WHEN TRAINING THE TEST DATASET
val_dict = df_val[categorical + numerical].to_dict(orient='records')
X_val    = dv.transform(val_dict)

In [24]:
# MAKE PREDICTIONS FOR THE VALIDATION DATASET BASED UPON THE TRAINING DATASET
y_pred   = model.predict_proba(X_val)[:, 1]

In [25]:
converted_decision = (y_pred >= 0.5)

In [26]:
# CHECK ACCURACY
(y_val == converted_decision).mean().round(2)

# ANSWER: .65

np.float64(0.64)

# Question 5

Let's find the least useful feature using the **feature elimination technique**. \
Train a model using the same features and parameters as in Q4 (without rounding). \
Now exclude each feature from this set and train a model without it. Record the accuracy for each model. \
For each feature, calculate the difference between the original accuracy and the accuracy without the feature. \
 \
Which of following feature has the smallest difference?
* 'lead_source'
* 'number_of_courses_viewed'
* 'interaction_count'

Note: The difference doesn't have to be positive.


In [41]:
# ONE-HOT ENCODING
#  IMPORT PACKAGE - PACKAGE IMPORTED FOR Q4
from sklearn.feature_extraction import DictVectorizer

#  CREATE DICTIONARY VECTOR
dv         = DictVectorizer(sparse=False)

#  ACCURACY FROM QUESTION 4
original_acc = (y_val == converted_decision).mean()
print(f"Original Accuracy\t\t{original_acc}")

Original Accuracy		0.645


In [45]:
# STRATEGY: 
#  1. BUILD A LOOP TO ITERATIVELY RE-CREATE THE FEATURE AND TARGET/LABEL DICTIONARIES
#  2. CREATE A NEW TRAINING VECTORIZER FOR EACH FEATURE-TARGET DICTIONARY PAIR
#  3. TRAIN THE MODELS AFTER DROPPING FEATURES
#  4. MAKE PREDICTIONS
#  5. EVALUATE ACCURACY

features_to_exclude = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for feat in features_to_exclude:
    # Create new train/val dictionaries without this feature
    train_dict_sub = [ {k: v for k, v in row.items() if k != feat} for row in train_dict ]
    val_dict_sub   = [ {k: v for k, v in row.items() if k != feat} for row in val_dict ]
    
    # Fit a new DictVectorizer for this specific feature subset
    dv_sub         = DictVectorizer(sparse=False)
    X_train_sub    = dv_sub.fit_transform(train_dict_sub)
    X_val_sub      = dv_sub.transform(val_dict_sub)
    
    # Train a new model without the feature
    model_sub      = LogisticRegression(solver='lbfgs', C=1.0, max_iter=5000, random_state=42)
    model_sub.fit(X_train_sub, y_train)
    
    # Predict and evaluate
    y_pred_sub     = model_sub.predict_proba(X_val_sub)[:, 1]
    decision_sub   = (y_pred_sub >= 0.5)
    acc_sub        = (y_val == decision_sub).mean()
    
    diff           = original_acc - acc_sub
    print(f"Dropping '{feat}': Accuracy = {acc_sub:.4f} \t|   Difference (Orig - Sub) = {diff:.4f}")

# ANSWER: 'lead_source' is more accurate (71.7% accurate)
#   Dropping 'lead_source': Accuracy = 0.7170 	            |   Difference (Orig - Sub) = -0.0720
#   Dropping 'number_of_courses_viewed': Accuracy = 0.7150 	|   Difference (Orig - Sub) = -0.0700
#   Dropping 'interaction_count': Accuracy = 0.7330 	    |   Difference (Orig - Sub) = -0.0880

Dropping 'lead_source': Accuracy = 0.7170 	|   Difference (Orig - Sub) = -0.0720
Dropping 'number_of_courses_viewed': Accuracy = 0.7150 	|   Difference (Orig - Sub) = -0.0700
Dropping 'interaction_count': Accuracy = 0.7330 	|   Difference (Orig - Sub) = -0.0880


# Question 6
Now let's train a regularized logistic regression. 
* Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
* Train models using all the features as in Q4. 
* Calculate the accuracy on the validation dataset and round it to 3 decimal digits. 

Which of these C leads to the best accuracy on the validation set?
* 0.000001
* 0.00001
* 0.0001
* 0.001


Note: If there are multiple options, select the smallest C.


In [46]:
from sklearn.linear_model import LogisticRegression

# List of C values to try
c_values = [0.000001, 0.00001, 0.0001, 0.001]

for c in c_values:
    # Train logistic regression with the given C parameter
    model = LogisticRegression(solver='lbfgs', C=c, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    # Make predictions and evaluate accuracy on the validation set
    y_pred = model.predict_proba(X_val)[:, 1]
    decision = (y_pred >= 0.5)
    accuracy = (y_val == decision).mean().round(3)
    
    print(f"C = {c:<9} | Validation Accuracy = {accuracy}")

C = 1e-06     | Validation Accuracy = 0.642
C = 1e-05     | Validation Accuracy = 0.649
C = 0.0001    | Validation Accuracy = 0.679
C = 0.001     | Validation Accuracy = 0.716
